In [86]:
!pip install bert-score transformers torch sentence-transformers numpy


[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Imports

In [87]:
import nltk
import json
import torch
import numpy as np
import torch.nn.functional as F

from bert_score import score
from transformers import logging
from sentence_transformers import SentenceTransformer
from transformers import BartTokenizer, BartForConditionalGeneration


In [88]:
logging.set_verbosity_error()

### Functions

In [89]:
def calculate_bertscore(paragraph:str, rewrite:str, type="bert"):
    if type == "bert":
        model = "bert-base-uncased"
    if type == "scibert":
        model = "allenai/scibert_scivocab_uncased"

    P, R, F1 = score(
    cands= rewrite,
    refs= paragraph,
    lang="en",
    model_type=model,
    verbose=False,
    )
    
    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

In [90]:
def calculate_bartscore(paragraph:str, rewrite:str):
    model_name = "facebook/bart-large"
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)
    model.eval()

    with torch.no_grad():
        inputs = tokenizer(paragraph, return_tensors="pt", truncation=True)
        labels = tokenizer(rewrite, return_tensors="pt", truncation=True)["input_ids"]

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss  # cross-entropy
        
        bart_score = -loss.item()

    return bart_score

In [91]:
def calculate_cosine_similarity(paragraph:str, rewrite:str):
    model_name = "sentence-transformers/all-mpnet-base-v2"
    model = SentenceTransformer(model_name)
    
    embeddings = model.encode([paragraph, rewrite], show_progress_bar=False)
    
    vec1 = embeddings[0]
    vec2 = embeddings[1]
    
    cosine = np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )
    return cosine

In [92]:
def calculate_metrics(paragraph, rewrite):
    bert_score = calculate_bertscore(paragraph, rewrite)
    #scibert_score = calculate_bertscore(paragraph, rewrite, "scibert")
    bart_score = calculate_bartscore(paragraph, rewrite)
    cosinus_similarity = calculate_cosine_similarity(paragraph[0], rewrite[0])
    return {"BertScore":bert_score,
            #"SciBertScore": scibert_score,
            "BartScore": bart_score,
            "Cos_Sim":cosinus_similarity}

In [93]:
with open("../data/json/rewrites.json","r" , encoding='utf-8') as file:
    data = json.load(file)

In [94]:
data = data[0]

In [95]:
calculate_metrics([data["Paragraphes"]], [data["Rew1"]])

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 194.39it/s, Materializing param=pooler.dense.weight]                        


{'BertScore': {'precision': 0.8747970461845398,
  'recall': 0.8725864887237549,
  'f1': 0.8736903071403503},
 'BartScore': -2.849609375,
 'Cos_Sim': np.float32(0.9189807)}